In [2]:
import pandas as pd

awards = pd.read_csv("data/Player Award Shares.csv")
stats = pd.read_csv("data/Player Per Game.csv")

print("Awards shape:", awards.shape)
print("Stats shape:", stats.shape)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print("AWARDS COLUMNS:")
print(awards.columns.tolist())

print("\nSTATS COLUMNS:")
print(stats.columns.tolist())

In [ ]:
print(awards["award"].unique())

In [ ]:
mvp = awards[awards["award"] == "nba mvp"]

print("MVP records:", len(mvp))
print("MVP winners:", mvp["winner"].sum())
print("Seasons:", mvp["season"].nunique())

In [ ]:
print(mvp["season"].min())
print(mvp["season"].max())

In [ ]:
print(mvp["winner"].dtype)
print(mvp["winner"].value_counts())

In [ ]:
print(mvp[mvp["winner"] == True][["season", "player", "pts_won", "share"]].head(10))

In [ ]:
duplicates = stats.duplicated(subset=["season", "player_id"], keep=False)

print("Duplicate player-season rows:", duplicates.sum())

In [ ]:
print(stats[stats["team"] == "TOT"].head())

In [ ]:
duplicate_rows = stats[stats.duplicated(
    subset=["season", "player_id"],
    keep=False
)]

print(duplicate_rows[[
    "season", "player", "team", "g",
    "fg_per_game", "fga_per_game",
    "trb_per_game", "ast_per_game", "pts_per_game"
]].head(10))

In [ ]:
print(stats[stats["team"].str.contains("TM", na=False)]["team"].value_counts())

In [ ]:
tm_rows = stats[stats["team"].str.contains("TM", na=False)]

print("Combined-team rows:", len(tm_rows))
print("Single-team rows:", len(stats[~stats["player_id"].isin(tm_rows["player_id"]) ]))

In [ ]:
tm_mask = stats["team"].str.contains("TM", na=False)

clean_stats = stats[
    tm_mask | ~stats.duplicated(subset=["season", "player_id"], keep=False)
].copy()

print("Original rows:", len(stats))
print("Clean rows:", len(clean_stats))
print("Duplicate player-seasons remaining:",
      clean_stats.duplicated(subset=["season", "player_id"]).sum())

In [ ]:
print("Stats seasons:", clean_stats["season"].min(), "to", clean_stats["season"].max())
print("MVP seasons:", mvp["season"].min(), "to", mvp["season"].max())

In [ ]:
clean_stats = clean_stats[
    (clean_stats["season"] >= 1956) &
    (clean_stats["season"] <= 2025)
].copy()

print("Clean stats shape:", clean_stats.shape)
print("Seasons:", clean_stats["season"].min(), "to", clean_stats["season"].max())

In [ ]:
mvp_winners = mvp[mvp["winner"] == True][["season", "player_id"]].copy()

mvp_winners["mvp"] = 1

data = clean_stats.merge(
    mvp_winners,
    on=["season", "player_id"],
    how="left"
)

data["mvp"] = data["mvp"].fillna(0).astype(int)

print("Final dataset shape:", data.shape)
print(data["mvp"].value_counts())

In [ ]:
players_per_season = data.groupby("season")["player_id"].nunique()

print(players_per_season.describe())

In [ ]:
print(data.groupby("season")["pts_per_game"].max().describe())

In [ ]:
mvp_sorted = mvp.sort_values(
    ["season", "pts_won"],
    ascending=[True, False]
)

print(mvp_sorted[[
    "season", "player", "pts_won", "winner"
]].tail(15))

In [ ]:
for threshold in [50, 55, 60, 65]:
    count = (data["g"] >= threshold).groupby(data["season"]).sum()
    print(f"G >= {threshold}: average candidates =", round(count.mean(), 1))

In [ ]:
eligible = data[data["g"] >= 50].copy()

print("Eligible rows:", len(eligible))
print("MVP winners:", eligible["mvp"].sum())

In [ ]:
print(data[(data["mvp"] == 1) & (data["g"] < 50)][
    ["season", "player", "g"]
])

In [ ]:
features = [
    "g",
    "mp_per_game",
    "fg_percent",
    "x3p_percent",
    "ft_percent",
    "trb_per_game",
    "ast_per_game",
    "stl_per_game",
    "blk_per_game",
    "pts_per_game"
]

X = data[features]
y = data["mvp"]

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
train = data[data["season"] <= 2019].copy()
test = data[data["season"] >= 2020].copy()

X_train = train[features]
y_train = train["mvp"]

X_test = test[features]
y_test = test["mvp"]

print("Training:", X_train.shape, y_train.sum())
print("Testing:", X_test.shape, y_test.sum())

In [ ]:
print("Training missing values:")
print(X_train.isna().sum())

print("\nTesting missing values:")
print(X_test.isna().sum())

In [ ]:
print(
    data.loc[data["x3p_percent"].isna(), "season"]
    .value_counts()
    .sort_index()
)

In [ ]:
missing_by_season = data.groupby("season")[features].apply(lambda x: x.isna().sum().sum())

print(missing_by_season[missing_by_season > 0])

In [ ]:
print(data.groupby("season")[features].apply(lambda x: x.isna().sum()).head(30))

In [ ]:
print("Training rows with any missing value:",
      X_train.isna().any(axis=1).sum())

print("Testing rows with any missing value:",
      X_test.isna().any(axis=1).sum())

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Training shape:", X_train_imputed.shape)
print("Testing shape:", X_test_imputed.shape)
print("Missing values after imputation:", X_train_imputed.shape[0] - X_train_imputed.shape[0])

In [ ]:
print("Training missing:", pd.isna(X_train_imputed).sum())
print("Testing missing:", pd.isna(X_test_imputed).sum())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Training:", X_train_scaled.shape)
print("Testing:", X_test_scaled.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train_scaled, y_train)

print("Logistic Regression trained successfully.")

In [ ]:
y_pred = logreg.predict(X_test_scaled)
y_prob = logreg.predict_proba(X_test_scaled)[:, 1]

print("Predictions made:", len(y_pred))
print("Predicted MVPs:", y_pred.sum())

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, digits=3))

In [ ]:
test_results = test[["season", "player", "player_id", "mvp"]].copy()

test_results["mvp_probability"] = y_prob

predicted_mvp = (
    test_results
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .first()
    .reset_index()
)

print(predicted_mvp[["season", "player", "mvp_probability", "mvp"]].to_string(index=False))

In [ ]:
season_accuracy = predicted_mvp["mvp"].mean()

print(f"Season-level MVP accuracy: {season_accuracy:.3f}")

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train_imputed, y_train)

print("Naive Bayes trained successfully.")

In [ ]:
y_pred_nb = nb.predict(X_test_imputed)
y_prob_nb = nb.predict_proba(X_test_imputed)[:, 1]

print("Predictions made:", len(y_pred_nb))
print("Predicted MVPs:", y_pred_nb.sum())

In [ ]:
test_results_nb = test[["season", "player", "player_id", "mvp"]].copy()

test_results_nb["mvp_probability"] = y_prob_nb

predicted_mvp_nb = (
    test_results_nb
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .first()
    .reset_index()
)

print(
    predicted_mvp_nb[
        ["season", "player", "mvp_probability", "mvp"]
    ].to_string(index=False)
)

In [ ]:
season_accuracy_nb = predicted_mvp_nb["mvp"].mean()

print(f"Naive Bayes season-level accuracy: {season_accuracy_nb:.3f}")

In [ ]:
logreg_coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": logreg.coef_[0]
})

logreg_coefficients["abs_coefficient"] = logreg_coefficients["coefficient"].abs()

print(
    logreg_coefficients
    .sort_values("abs_coefficient", ascending=False)
    .to_string(index=False)
)

In [ ]:
top5_logreg = (
    test_results
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .head(5)
)

print(
    top5_logreg[
        ["season", "player", "mvp_probability", "mvp"]
    ].to_string(index=False)
)

In [ ]:
top3_logreg = (
    test_results
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .head(3)
)

top3_accuracy = (
    top3_logreg.groupby("season")["mvp"].max().mean()
)

print(f"Logistic Regression Top-3 accuracy: {top3_accuracy:.3f}")

In [ ]:
top3_nb = (
    test_results_nb
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .head(3)
)

top3_accuracy_nb = (
    top3_nb.groupby("season")["mvp"].max().mean()
)

print(f"Naive Bayes Top-3 accuracy: {top3_accuracy_nb:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

print("Random Forest trained successfully.")

In [ ]:
y_pred_rf = rf.predict(X_test_imputed)
y_prob_rf = rf.predict_proba(X_test_imputed)[:, 1]

print("Predictions made:", len(y_pred_rf))
print("Predicted MVPs:", y_pred_rf.sum())

In [ ]:
test_results_rf = test[["season", "player", "player_id", "mvp"]].copy()

test_results_rf["mvp_probability"] = y_prob_rf

predicted_mvp_rf = (
    test_results_rf
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .first()
    .reset_index()
)

print(
    predicted_mvp_rf[
        ["season", "player", "mvp_probability", "mvp"]
    ].to_string(index=False)
)

In [ ]:
top3_rf = (
    test_results_rf
    .sort_values(["season", "mvp_probability"], ascending=[True, False])
    .groupby("season")
    .head(3)
)

top3_accuracy_rf = (
    top3_rf.groupby("season")["mvp"].max().mean()
)

print(f"Random Forest Top-3 accuracy: {top3_accuracy_rf:.3f}")

In [ ]:
print(
    top3_rf.groupby("season")["mvp"].max()
)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf, digits=3))

In [ ]:
rf_importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
})

print(
    rf_importance
    .sort_values("importance", ascending=False)
    .to_string(index=False)
)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Random Forest"
    ],
    "Top-1 Accuracy": [
        0.167,
        0.167,
        0.500
    ],
    "Top-3 Accuracy": [
        0.667,
        0.667,
        0.833
    ]
})

print(comparison.to_string(index=False))

In [ ]:
def get_actual_mvp_rank(results):
    ranks = []

    for season in sorted(results["season"].unique()):
        season_data = results[results["season"] == season].sort_values(
            "score", ascending=False
        ).reset_index(drop=True)

        actual = season_data[season_data["mvp"] == 1]

        if len(actual) > 0:
            rank = actual.index[0] + 1
            ranks.append([season, actual.iloc[0]["player"], rank])

    return pd.DataFrame(ranks, columns=["season", "actual_mvp", "rank"])


logreg_rank = test_results.copy()
logreg_rank["score"] = y_prob

nb_results = test[["season", "player", "player_id", "mvp"]].copy()
nb_results["score"] = nb.predict_proba(X_test_imputed)[:, 1]

rf_results = test[["season", "player", "player_id", "mvp"]].copy()
rf_results["score"] = rf.predict_proba(X_test_imputed)[:, 1]

print("LOGISTIC REGRESSION")
print(get_actual_mvp_rank(logreg_rank))

print("\nNAIVE BAYES")
print(get_actual_mvp_rank(nb_results))

print("\nRANDOM FOREST")
print(get_actual_mvp_rank(rf_results))

In [ ]:
rf_2023 = rf_results[rf_results["season"] == 2023].copy()

rf_2023 = rf_2023.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

rf_2023["rank"] = rf_2023.index + 1

print(
    rf_2023[
        ["rank", "player", "score", "mvp"]
    ].head(10).to_string(index=False)
)

In [ ]:
ppg_results = test[["season", "player", "player_id", "pts_per_game", "mvp"]].copy()

ppg_top1 = (
    ppg_results
    .sort_values(["season", "pts_per_game"], ascending=[True, False])
    .groupby("season")
    .first()
    .reset_index()
)

ppg_top3 = (
    ppg_results
    .sort_values(["season", "pts_per_game"], ascending=[True, False])
    .groupby("season")
    .head(3)
)

ppg_top1_accuracy = ppg_top1["mvp"].mean()

ppg_top3_accuracy = (
    ppg_top3.groupby("season")["mvp"]
    .max()
    .mean()
)

print("PPG Baseline Top-1 Accuracy:", round(ppg_top1_accuracy, 3))
print("PPG Baseline Top-3 Accuracy:", round(ppg_top3_accuracy, 3))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance.to_string(index=False))

In [ ]:
print("Available seasons:")
print(sorted(data["season"].unique())[-20:])

In [ ]:
walk_forward_seasons = list(range(2010, 2026))

print("Walk-forward test seasons:")
print(walk_forward_seasons)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

walk_forward_results = []

for test_season in walk_forward_seasons:
    train_data = data[data["season"] < test_season].copy()
    test_data = data[data["season"] == test_season].copy()

    X_train_wf = train_data[features]
    y_train_wf = train_data["mvp"]

    X_test_wf = test_data[features]

    imputer_wf = SimpleImputer(strategy="median")

    X_train_wf = imputer_wf.fit_transform(X_train_wf)
    X_test_wf = imputer_wf.transform(X_test_wf)

    rf_wf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_wf.fit(X_train_wf, y_train_wf)

    test_data["score"] = rf_wf.predict_proba(X_test_wf)[:, 1]

    test_data = test_data.sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)

    test_data["rank"] = test_data.index + 1

    actual_mvp = test_data[test_data["mvp"] == 1]

    if len(actual_mvp) > 0:
        walk_forward_results.append({
            "season": test_season,
            "actual_mvp": actual_mvp.iloc[0]["player"],
            "mvp_rank": actual_mvp.iloc[0]["rank"]
        })

walk_forward_results = pd.DataFrame(walk_forward_results)

print(walk_forward_results.to_string(index=False))

In [ ]:
print(
    data.groupby("season").size().loc[2010:2025]
)

In [ ]:
candidate_data = data[data["g"] >= 50].copy()

print("Original rows:", len(data))
print("Candidate rows:", len(candidate_data))

print("\nPlayers per season:")
print(
    candidate_data.groupby("season").size().loc[2010:2025]
)

In [ ]:
mvp_check = data[data["mvp"] == 1][
    ["season", "player", "g", "pts_per_game"]
].copy()

print(mvp_check.sort_values("g").head(15).to_string(index=False))

In [ ]:
mvp_minutes = data[data["mvp"] == 1][
    ["season", "player", "g", "mp_per_game", "pts_per_game"]
].copy()

print(
    mvp_minutes.sort_values("mp_per_game").head(15).to_string(index=False)
)

In [ ]:
mvp_awards = awards[awards["award"] == "nba mvp"].copy()

print(
    mvp_awards[
        ["season", "player", "pts_won", "pts_max", "share", "winner"]
    ].sort_values(
        ["season", "pts_won"],
        ascending=[False, False]
    ).head(20).to_string(index=False)
)

In [ ]:
vote_counts = (
    mvp_awards[mvp_awards["pts_won"] > 0]
    .groupby("season")
    .size()
)

print(vote_counts.loc[2010:2025])
print("\nAverage vote-getters:", round(vote_counts.mean(), 2))
print("Minimum:", vote_counts.min())
print("Maximum:", vote_counts.max())

In [ ]:
evaluation_candidates = mvp_awards[
    mvp_awards["pts_won"] > 0
][["season", "player_id", "player", "pts_won", "winner"]].copy()

print("Evaluation rows:", len(evaluation_candidates))

print(
    evaluation_candidates
    .groupby("season")
    .size()
    .loc[2010:2025]
)

In [ ]:
rf_candidate_eval = rf_results.merge(
    evaluation_candidates[["season", "player_id", "pts_won", "winner"]],
    on=["season", "player_id"],
    how="inner"
)

print("RF evaluation rows:", len(rf_candidate_eval))

print(
    rf_candidate_eval[
        ["season", "player", "score", "pts_won", "winner"]
    ].head(20).to_string(index=False)
)

In [ ]:
rf_vote_rank = []

for season in sorted(rf_candidate_eval["season"].unique()):
    season_data = rf_candidate_eval[
        rf_candidate_eval["season"] == season
    ].sort_values("score", ascending=False).reset_index(drop=True)

    season_data["rank"] = season_data.index + 1

    actual_mvp = season_data[season_data["winner"] == True]

    if len(actual_mvp) > 0:
        rf_vote_rank.append({
            "season": season,
            "actual_mvp": actual_mvp.iloc[0]["player"],
            "rank_among_vote_getters": actual_mvp.iloc[0]["rank"],
            "vote_getters": len(season_data)
        })

rf_vote_rank = pd.DataFrame(rf_vote_rank)

print(rf_vote_rank.to_string(index=False))

In [ ]:
print("Average MVP rank:", round(rf_vote_rank["rank_among_vote_getters"].mean(), 2))
print("Median MVP rank:", rf_vote_rank["rank_among_vote_getters"].median())
print("Top-1 accuracy:", round((rf_vote_rank["rank_among_vote_getters"] == 1).mean(), 3))
print("Top-3 accuracy:", round((rf_vote_rank["rank_among_vote_getters"] <= 3).mean(), 3))

In [ ]:
def rank_among_vote_getters(model_results):
    merged = model_results.merge(
        evaluation_candidates[["season", "player_id", "winner"]],
        on=["season", "player_id"],
        how="inner"
    )

    results = []

    for season in sorted(merged["season"].unique()):
        season_data = merged[
            merged["season"] == season
        ].sort_values("score", ascending=False).reset_index(drop=True)

        season_data["rank"] = season_data.index + 1

        actual_mvp = season_data[season_data["winner"] == True]

        if len(actual_mvp) > 0:
            results.append({
                "season": season,
                "actual_mvp": actual_mvp.iloc[0]["player"],
                "rank": actual_mvp.iloc[0]["rank"]
            })

    return pd.DataFrame(results)


logreg_vote_rank = rank_among_vote_getters(logreg_rank)
nb_vote_rank = rank_among_vote_getters(nb_results)

print("LOGISTIC REGRESSION")
print(logreg_vote_rank.to_string(index=False))

print("\nNAIVE BAYES")
print(nb_vote_rank.to_string(index=False))

In [ ]:
model_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Random Forest"
    ],
    "Average Rank": [
        logreg_vote_rank["rank"].mean(),
        nb_vote_rank["rank"].mean(),
        rf_vote_rank["rank_among_vote_getters"].mean()
    ],
    "Median Rank": [
        logreg_vote_rank["rank"].median(),
        nb_vote_rank["rank"].median(),
        rf_vote_rank["rank_among_vote_getters"].median()
    ],
    "Top-1 Accuracy": [
        (logreg_vote_rank["rank"] == 1).mean(),
        (nb_vote_rank["rank"] == 1).mean(),
        (rf_vote_rank["rank_among_vote_getters"] == 1).mean()
    ],
    "Top-3 Accuracy": [
        (logreg_vote_rank["rank"] <= 3).mean(),
        (nb_vote_rank["rank"] <= 3).mean(),
        (rf_vote_rank["rank_among_vote_getters"] <= 3).mean()
    ]
})

print(model_summary.round(3).to_string(index=False))

In [ ]:
chart_data = model_summary.copy()

chart_data["Top-1 Accuracy"] = chart_data["Top-1 Accuracy"] * 100
chart_data["Top-3 Accuracy"] = chart_data["Top-3 Accuracy"] * 100

print(chart_data.to_string(index=False))

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("results", exist_ok=True)

# Model comparison graph
models = model_summary["Model"]
top1 = model_summary["Top-1 Accuracy"] * 100
top3 = model_summary["Top-3 Accuracy"] * 100

x = range(len(models))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar([i - width/2 for i in x], top1, width, label="Top-1 Accuracy")
plt.bar([i + width/2 for i in x], top3, width, label="Top-3 Accuracy")
plt.xticks(list(x), models)
plt.ylabel("Accuracy (%)")
plt.title("NBA MVP Prediction: Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("results/model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Feature importance graph
plt.figure(figsize=(9, 5))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("results/feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

print("Both graphs saved successfully!")